# Linear Regression with Neural Networks
**Student:** Daniel Sozoranga

**Subject:** Machine Learning

**Assignment:** Make-up work (extra points)

## Goal
To predict the yearly cost of medical insurance using personal details (age, BMI, children, smoker, region, sex) using neural networks with TensorFlow/Keras.

We trained 3 models on the same problem to compare how they learn:
1. **Well-trained**: good balance (bias/variance), using Dropout.
2. **Underfitted**: the model fails to learn enough.
3. **Overfitted**: the model memorizes the training data but fails on the test data.

## Short Theory
**Regression** predicts a continuous number. A neural network does regression when the last layer has only one neuron with linear activation. Without hidden layers, it is the same as basic linear regression; with hidden layers, it learns more complex patterns.

### The Cake Analogy
- **Features (ingredients):** age, BMI, smoker, etc.
- **Parameters (recipe):** the weights that the network adjusts in each round (epoch).
- **Epochs:** each full pass through the data.
- **Underfitting:** simple recipe, raw cake.
- **Overfitting:** we memorize THAT specific cake, but we don't know how to bake a different one.
- **Well-trained:** a general recipe, the cake is always delicious.

## 1. Imports

**Libraries.** `numpy`/`pandas` for handling numbers and data tables, `matplotlib` for graphs, `tensorflow/keras` for building and training the neural networks, and `sklearn` tools for splitting data (train/test), scaling, measuring results, checking accuracy (cross-validation), and understanding the model. A random seed (42) is set so we get the exact same results every time we run the code.

In [ ]:
# === LIBRARIES ===
import numpy as np                  # numeric computing and multi-dim arrays
import pandas as pd                 # tables (DataFrames) to load and explore data
import matplotlib.pyplot as plt     # plots (loss curves, residuals, etc.)
import tensorflow as tf             # deep learning framework
from tensorflow import keras        # high-level TF API to build networks
from tensorflow.keras import layers, regularizers  # layers (Dense, Dropout) and regularizers
from sklearn.model_selection import train_test_split, KFold, learning_curve, validation_curve  # train/test split and cross-validation
from sklearn.preprocessing import StandardScaler   # standardization (mean 0, std 1)
from sklearn.linear_model import LinearRegression  # baseline model to compare against the NN
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # regression metrics
from sklearn.inspection import permutation_importance  # permutation-based feature importance
from sklearn.base import BaseEstimator, RegressorMixin  # to wrap keras models into sklearn API

np.random.seed(42); tf.random.set_seed(42)  # seeds for reproducibility across runs


## 2. Loading the dataset

**Dataset:** We load the well-known US medical insurance dataset (1,338 records). The target `charges` is a continuous number → this is a regression problem. Features: age, sex, BMI, number of children, smoker/non-smoker, and region.

In [ ]:
# URL of the public medical insurance dataset (1338 rows, 7 columns)
URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(URL)               # download the CSV into a DataFrame
print("Shape:", df.shape)     # print shape to verify the load
df.head()                            # show the first 5 rows


**Descriptive statistics:** They help us see that the variables have very different scales (like age 18–64 vs. charges 1k–64k). This shows why we need to standardize the features and transform the target to fix its uneven shape.

In [ ]:
df.describe()   # basic stats (count, mean, std, min, quartiles, max) of numeric columns


## 3. Preprocessing
One-hot encoding for categorical variables, 80/20 split, and standardization. We apply `log1p` on the target to fix its uneven shape (asymmetry).

**Preprocessing:** We change the categorical variables into numbers using *one-hot encoding* (`drop_first=True` prevents them from perfectly overlapping). We randomly split the data into 80% for training and 20% for testing, using a fixed seed. We apply `StandardScaler` (mean 0, standard deviation 1) to the features. We fit this scaler only on the training data to avoid *data leakage* (accidentally learning from the test data). We apply `log1p` to the target because the `charges` data is very lopsided (strongly skewed to the right): the log transformation makes the shape more even and makes the network much easier to train.

In [ ]:
# one-hot encode categorical columns; drop_first=True avoids perfect collinearity
df_enc = pd.get_dummies(df, columns=['sex','smoker','region'], drop_first=True).astype(float)
X = df_enc.drop('charges', axis=1).values   # feature matrix (all columns except the target)
y = df_enc['charges'].values                # target vector: yearly cost in USD
feat_names = df_enc.drop('charges', axis=1).columns.tolist()  # feature names (used later in permutation importance)
y_log = np.log1p(y)                         # log(1+y) transform to fix target skewness

# 80/20 split with fixed random seed
X_tr, X_te, y_tr_log, y_te_log, y_tr, y_te = train_test_split(
    X, y_log, y, test_size=0.2, random_state=42)  # random_state ensures the split is always the same

scaler = StandardScaler()           # scaler: learns mean and std
X_tr_s = scaler.fit_transform(X_tr) # fit on TRAIN only and transform (IMPORTANT: no leakage)
X_te_s = scaler.transform(X_te)     # transform TEST with the train parameters
print("Train:", X_tr_s.shape, " Test:", X_te_s.shape)  # check shapes


## 4. Loss function and metrics
- **MSE (loss):** differentiable, penalizes large errors, standard in regression.
- **MAE:** average absolute error in USD, handles outliers well.
- **RMSE:** √MSE, easy to understand because it uses the target's original units.
- **R²:** the percentage of variance explained (how well the model fits).
- **Robust alternative:** Huber Loss (squared near 0, linear further away). Very useful if there are extreme outliers.

## 5. Baseline model: Linear Regression (reported with and without log1p)
A fair comparison to show that the Neural Network adds real value over the basic model.

**Baseline model:** Linear Regression. We train a classic linear model as a *baseline* (a starting point). Using a neural network is only worth it if it clearly beats this starting point; if not, it means there are no important non-linear patterns to learn.

In [ ]:
# === BASELINE 1: linear regression on y in original scale (USD) ===
lin_raw = LinearRegression().fit(X_tr_s, y_tr)   # fit the classic linear model
pred_lin_raw = lin_raw.predict(X_te_s)            # predictions on test
print(f"LR on raw y -> MAE={mean_absolute_error(y_te,pred_lin_raw):,.2f} | "
      f"RMSE={np.sqrt(mean_squared_error(y_te,pred_lin_raw)):,.2f} | R2={r2_score(y_te,pred_lin_raw):.4f}")

# === BASELINE 2: linear regression on log1p(y), reversed with expm1 ===
lin_log = LinearRegression().fit(X_tr_s, y_tr_log)   # fit in log scale
pred_lin_log = np.expm1(lin_log.predict(X_te_s))     # expm1 reverses log1p -> back to USD
print(f"LR on log1p(y) -> MAE={mean_absolute_error(y_te,pred_lin_log):,.2f} | "
      f"RMSE={np.sqrt(mean_squared_error(y_te,pred_lin_log)):,.2f} | R2={r2_score(y_te,pred_lin_log):.4f}")

pred_lin = pred_lin_raw   # use the raw-scale baseline for the final table
